In [ ]:
from models.mae import MaskedAutoEncoder
from torch.utils.data import DataLoader
from torch import optim
import torch


device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model = MaskedAutoEncoder()

In [ ]:
from torchvision.transforms import v2
from medmnist import PathMNIST

tf = v2.Compose([
    v2.ToTensor(),
    #v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = PathMNIST(root="./data/",split="train",transform=tf,download=True,size=64)
val_dataset = PathMNIST(root="./data/",split="val",transform=tf,download=True,size=64)

In [ ]:

from torch.utils.data import RandomSampler

num_workers = 4

train_batch_size = 256
val_batch_size = 256
effective_batch = 256

train_dl = DataLoader(
    train_dataset,
    batch_size= train_batch_size,
    shuffle=True, 
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
)

val_dl = DataLoader(
    val_dataset,
    batch_size = val_batch_size,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
)

sample = RandomSampler(val_dataset,num_samples=1)
sample_loader = DataLoader(val_dataset, batch_size=1, sampler=sample)

In [ ]:
grad_acc = max(1,(256//train_batch_size))
steps_per_epoch = len(train_dl)

epochs = 800
warm_up_epochs = int(epochs * 0.025)
checkpointing_rate = 25
show_visuals_rate = 5

warm_up_steps = warm_up_epochs*steps_per_epoch
cosine_steps = (epochs - warm_up_epochs)*steps_per_epoch

base_lr = 1.5e-3
max_lr = base_lr * (effective_batch/256)
min_lr = 1e-6

optimiser = optim.AdamW(
    filter(lambda p: p.requires_grad,
           model.parameters()),
           lr=max_lr,betas=(0.9, 0.95),
           weight_decay=0.05)

# start at minimum and go up
warmup_scheduler = optim.lr_scheduler.LinearLR(
    optimiser, start_factor=(min_lr/max_lr), end_factor=1.0, total_iters=warm_up_steps
)
cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimiser, T_max=cosine_steps, eta_min=min_lr
)
scheduler = optim.lr_scheduler.SequentialLR(
    optimiser, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warm_up_steps]
)


In [ ]:
from pretraining_functions import train,test
from tqdm.notebook import tqdm
import time
import csv
import matplotlib.pyplot as plt
import numpy as np

log_file_path = f"logs/v3_pretraining_log_{time.time()}.csv"

with open(log_file_path, mode="w", newline="") as f:
    prev_train_loss = None
    prev_val_loss = None
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss","train_delta","val_delta","current_lr"])
    model = model.to(device)
    torch.set_float32_matmul_precision('high')
    model = torch.compile(model)
    with tqdm(range(epochs),desc="Epochs") as bar:
        for epoch in bar:
            train_delta = val_delta = 0.0
            train_loss, current_lr = train(model, device, train_dl, optimiser, epoch, scheduler, grad_acc)
            val_loss = test(model,device,val_dl, epoch)
            train_delta = train_loss - prev_train_loss  if prev_train_loss is not None else 0.0
            val_delta = val_loss - prev_val_loss if prev_val_loss is not None else 0.0
            # quick visual check that training is work to get better at reconstruction
            if (epoch+1) % show_visuals_rate == 0:
                for sample,_ in sample_loader:
                    sample = sample.to(device)
                    reconstructed_sample,loss_mask,_ = model(sample)
                    original_image = np.clip(np.reshape(sample.detach().cpu(),(3,64,64)).permute(1, 2, 0), 0, 1)
                    visible_patches = np.clip(np.reshape(sample.detach().cpu()*(-1*(loss_mask.detach().cpu()-torch.ones(loss_mask.shape))),(3,64,64)).permute(1, 2, 0), 0, 1)
                    reconstructed_patches = np.clip(np.reshape(reconstructed_sample.detach().cpu()*loss_mask.detach().cpu(),(3,64,64)).permute(1, 2, 0), 0, 1)
                    fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(16, 4))
                    axes.flat[0].imshow(original_image)
                    axes.flat[0].set_title("Original Image")
                    axes.flat[1].imshow(visible_patches)
                    axes.flat[1].set_title("Visible Patches")
                    axes.flat[2].imshow(reconstructed_patches)
                    axes.flat[2].set_title("Reconstructed Patches")
                    axes.flat[3].imshow(reconstructed_patches+visible_patches)
                    axes.flat[3].set_title("Reconstructed Image")
                    stats_title = (f"Epoch {epoch+1}   |   Training Loss: {train_loss:.3e}   |   Validation Loss: {val_loss:.3e}\n"
                                    f"Learning Rate: {current_lr:.3e}   |   Training Delta: {train_delta:+.3e}   |   Validation Delta: {val_delta:+.3e}")
                    plt.suptitle(stats_title, fontsize=12, fontweight='bold', y=1.12)
                    plt.tight_layout()
                    plt.show()
                    plt.close(fig)

            prev_train_loss = train_loss
            prev_val_loss = val_loss

            bar.set_postfix({
                    "train_loss": f"{train_loss:.3e}",
                    "train_delta":f"{train_delta:+.3e}",
                    "val_loss": f"{val_loss:.3e}",
                    "val_delta":f"{val_delta:+.3e}",
                    "lr": f"{current_lr:.3e}"
                })
            writer.writerow([epoch+1, train_loss, val_loss, train_delta, val_delta, current_lr])
            f.flush()

            if (epoch+1) % checkpointing_rate == 0:
                torch.save(model._orig_mod.state_dict() if hasattr(model, '_orig_mod') else model.state_dict(),f"model_weights/pretrain_checkpoints/model_13_epoch_{epoch+1}.pt")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import time 

dataframe = pd.read_csv(log_file_path)

plt.plot(dataframe["epoch"],dataframe["train_loss"])
plt.plot(dataframe["epoch"],dataframe["val_loss"])
plt.title
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.savefig(f"graphs/v3_pretraining/loss_{time.time()}.png")
plt.show()
plt.close()